# 01. Khám phá chi tiết các biến RFM do `rfm-fe` tạo ra

Notebook này minh họa:
1. Cấu trúc DataFrame giao dịch đầu vào `df`.
2. Toàn bộ các biến mới được `RFMFeatureEngineer` sinh ra (chia theo các nhóm).
3. Đối chiếu từng con số của 1 khách hàng cụ thể (`CUST_00001`) từ giao dịch gốc sang bảng RFM.
4. Thống kê mô tả toàn bộ các biến mới.

In [1]:
import sys
from pathlib import Path

# Đảm bảo import được package từ thư mục gốc repo
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from rfm_fe import RFMFeatureEngineer, engineer_rfm_features

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

## 1. Tạo dữ liệu giao dịch mẫu `df` (100.000 dòng, 10.000 khách hàng)

In [2]:
def generate_rfm_test_data(n_rows=100_000, n_customers=10_000, seed=42):
    rng = np.random.default_rng(seed)
    weights = 1.0 / np.arange(1, n_customers + 1) ** 0.8
    weights /= weights.sum()
    customer_ids = rng.choice(
        [f"CUST_{i:05d}" for i in range(1, n_customers + 1)],
        size=n_rows,
        p=weights,
    )
    # Cho phép nhiều dòng thuộc cùng 1 hóa đơn để test transaction_col
    invoice_ids = [f"INV_{i:07d}" for i in rng.integers(1, n_rows // 2 + 1, size=n_rows)]
    end_date = np.datetime64("2026-11-27")
    random_days = rng.integers(0, 365, size=n_rows)
    random_seconds = rng.integers(0, 86400, size=n_rows)
    invoice_dates = (
        end_date
        - random_days.astype("timedelta64[D]")
        - random_seconds.astype("timedelta64[s]")
    )
    quantities = rng.integers(1, 15, size=n_rows)
    unit_prices = np.round(rng.lognormal(mean=11.5, sigma=0.9, size=n_rows), -3)
    total_amounts = quantities * unit_prices
    statuses = rng.choice(["Completed", "Cancelled", "Refunded"], size=n_rows, p=[0.97, 0.02, 0.01])
    total_amounts = np.where(statuses == "Refunded", -total_amounts, total_amounts)
    categories = rng.choice(["Electronics", "Fashion", "Home", "Beauty", "Grocery"], size=n_rows)
    df = pd.DataFrame({
        "invoice_id": invoice_ids,
        "customer_id": customer_ids,
        "invoice_date": invoice_dates,
        "quantity": quantities,
        "unit_price": unit_prices,
        "total_amount": total_amounts,
        "category": categories,
        "status": statuses,
    })
    return df.sort_values("invoice_date").reset_index(drop=True)

df = generate_rfm_test_data()
print("Input shape:", df.shape)
df.head()

Input shape: (100000, 8)


,invoice_id,customer_id,invoice_date,quantity,unit_price,total_amount,category,status
0,INV_0022990,CUST_01139,2025-11-27 00:07:21,13,71000.0,923000.0,Electronics,Completed
1,INV_0031501,CUST_00080,2025-11-27 00:10:21,10,59000.0,590000.0,Grocery,Completed
2,INV_0002190,CUST_00005,2025-11-27 00:13:26,6,150000.0,900000.0,Beauty,Completed
3,INV_0021790,CUST_00655,2025-11-27 00:22:33,1,176000.0,176000.0,Home,Completed
4,INV_0045263,CUST_00193,2025-11-27 00:32:45,5,20000.0,100000.0,Beauty,Completed


## 2. Chạy `RFMFeatureEngineer` và liệt kê toàn bộ biến mới được tạo ra

In [3]:
df_valid = df[df["status"] != "Cancelled"].copy()

fe = RFMFeatureEngineer(
    customer_col="customer_id",
    date_col="invoice_date",
    amount_col="total_amount",
    transaction_col="invoice_id",
    category_col="category",
    reference_date="2026-11-28",
    windows=(30, 90, 180, 365),
    handle_negative_amount="separate",
    include_segment_label=True,
    encode_segment=True,
)
features_df = fe.fit_transform(df_valid)
feature_names = fe.get_feature_names_out().tolist()
print(f"Output shape: {features_df.shape} (gồm customer_id + {len(feature_names)} biến mới)")
print("\nDanh sách toàn bộ biến mới:")
for idx, col in enumerate(feature_names, 1):
    print(f"{idx:2d}. {col}")

Output shape: (9685, 61) (gồm customer_id + 60 biến mới)

Danh sách toàn bộ biến mới:
 1. monetary_sum
 2. monetary_mean
 3. monetary_std
 4. monetary_min
 5. monetary_max
 6. monetary_median
 7. frequency
 8. recency_days
 9. tenure_days
10. active_span_days
11. avg_order_value
12. monetary_cv
13. monetary_range
14. frequency_per_month
15. monetary_per_month
16. recency_to_tenure_ratio
17. is_repeat_customer
18. freq_30d
19. monetary_sum_30d
20. monetary_mean_30d
21. freq_90d
22. monetary_sum_90d
23. monetary_mean_90d
24. freq_180d
25. monetary_sum_180d
26. monetary_mean_180d
27. freq_365d
28. monetary_sum_365d
29. monetary_mean_365d
30. freq_velocity_30d_vs_365d
31. monetary_velocity_30d_vs_365d
32. ipt_mean_days
33. ipt_std_days
34. ipt_min_days
35. ipt_max_days
36. ipt_last_days
37. weekend_purchase_ratio
38. preferred_hour_mean
39. active_months_count
40. category_nunique
41. category_top_share
42. category_entropy
43. refund_count
44. refund_amount_sum
45. refund_rate
46. r_score

## 3. Quan sát giá trị từng nhóm biến mới

In [4]:
groups = {
    "1_base_rfm": ["customer_id", "recency_days", "frequency", "monetary_sum", "monetary_mean", "monetary_std", "monetary_min", "monetary_max", "monetary_median", "tenure_days", "active_span_days"],
    "2_ratios": ["customer_id", "avg_order_value", "monetary_cv", "monetary_range", "frequency_per_month", "monetary_per_month", "recency_to_tenure_ratio", "is_repeat_customer"],
    "3_windows_velocity": ["customer_id"] + [c for c in feature_names if c.startswith(("freq_", "monetary_sum_", "monetary_mean_", "monetary_velocity_"))],
    "4_interpurchase": ["customer_id"] + [c for c in feature_names if c.startswith("ipt_")],
    "5_temporal_category_refund": ["customer_id", "weekend_purchase_ratio", "preferred_hour_mean", "active_months_count", "category_nunique", "category_top_share", "category_entropy", "refund_count", "refund_amount_sum", "refund_rate"],
    "6_scores_segments": ["customer_id", "r_score", "f_score", "m_score", "rfm_score_sum", "rfm_score_weighted", "rfm_segment"],
}

for name, cols in groups.items():
    print(f"\n=== {name} ===")
    display(features_df[cols].head(5))


=== 1_base_rfm ===


,customer_id,recency_days,frequency,monetary_sum,monetary_mean,monetary_std,monetary_min,monetary_max,monetary_median,tenure_days,active_span_days
0,CUST_00001,1.128750,3513.0,4.141095e+09,1.141112e+06,1.544243e+06,10000.0,20410000.0,654000.0,365.974259,364.845509
1,CUST_00002,1.291933,1988.0,2.276807e+09,1.122133e+06,1.717743e+06,9000.0,33628000.0,638000.0,365.923889,364.631956
2,CUST_00003,1.035012,1401.0,1.495749e+09,1.046710e+06,1.300909e+06,4000.0,15106000.0,670000.0,365.830093,364.795081
3,CUST_00004,1.173032,1175.0,1.280396e+09,1.077774e+06,1.486261e+06,15000.0,20124000.0,622000.0,365.885266,364.712234
4,CUST_00005,1.371273,925.0,9.846000e+08,1.048562e+06,1.321720e+06,5000.0,13932000.0,612000.0,365.990671,364.619398



=== 2_ratios ===


,customer_id,avg_order_value,monetary_cv,monetary_range,frequency_per_month,monetary_per_month,recency_to_tenure_ratio,is_repeat_customer
0,CUST_00001,1.178792e+06,1.353279,20400000.0,287.971073,3.394579e+08,0.003084,1.0
1,CUST_00002,1.145275e+06,1.530784,33619000.0,162.984713,1.866623e+08,0.003531,1.0
2,CUST_00003,1.067630e+06,1.242855,15102000.0,114.889400,1.226593e+08,0.002829,1.0
3,CUST_00004,1.089699e+06,1.379009,20109000.0,96.341677,1.049834e+08,0.003206,1.0
4,CUST_00005,1.064432e+06,1.260507,13927000.0,75.821605,8.070698e+07,0.003747,1.0



=== 3_windows_velocity ===


,customer_id,freq_30d,monetary_sum_30d,monetary_mean_30d,freq_90d,monetary_sum_90d,monetary_mean_90d,freq_180d,monetary_sum_180d,monetary_mean_180d,freq_365d,monetary_sum_365d,monetary_mean_365d,freq_velocity_30d_vs_365d,monetary_velocity_30d_vs_365d
0,CUST_00001,265.0,293911000.0,1.109098e+06,874.0,986835000.0,1.115068e+06,1719.0,1.978084e+09,1.126472e+06,3502.0,4.126757e+09,1.140618e+06,0.920664,0.866520
1,CUST_00002,164.0,168221000.0,1.025738e+06,508.0,525547000.0,1.030484e+06,975.0,1.057270e+09,1.073371e+06,1979.0,2.269705e+09,1.123616e+06,1.008253,0.901742
2,CUST_00003,125.0,164873000.0,1.318984e+06,354.0,403559000.0,1.136786e+06,715.0,7.994560e+08,1.105748e+06,1394.0,1.491050e+09,1.048558e+06,1.090985,1.345330
3,CUST_00004,85.0,109818000.0,1.291976e+06,294.0,337321000.0,1.147350e+06,576.0,6.256110e+08,1.082372e+06,1172.0,1.270764e+09,1.072375e+06,0.882395,1.051430
4,CUST_00005,76.0,63292000.0,8.327895e+05,228.0,216453000.0,9.452096e+05,465.0,4.875520e+08,1.041778e+06,922.0,9.799800e+08,1.046987e+06,1.002892,0.785784


... (2 outputs left out)


=== 5_temporal_category_refund ===


,customer_id,weekend_purchase_ratio,preferred_hour_mean,active_months_count,category_nunique,category_top_share,category_entropy,refund_count,refund_amount_sum,refund_rate
0,CUST_00001,0.271425,11.397079,13.0,5.0,0.209149,2.321316,41.0,35126000.0,0.011536
1,CUST_00002,0.290784,11.633317,13.0,5.0,0.203549,2.321829,22.0,27873000.0,0.010945
2,CUST_00003,0.284115,11.485654,13.0,5.0,0.214136,2.319902,13.0,8459000.0,0.009194
3,CUST_00004,0.287037,11.303030,13.0,5.0,0.212121,2.321052,13.0,13056000.0,0.010943
4,CUST_00005,0.312034,11.400426,13.0,5.0,0.212993,2.319617,11.0,14589000.0,0.011752



=== 6_scores_segments ===


,customer_id,r_score,f_score,m_score,rfm_score_sum,rfm_score_weighted,rfm_segment
0,CUST_00001,5.0,5.0,5.0,15.0,15.0,Champions
1,CUST_00002,5.0,5.0,5.0,15.0,15.0,Champions
2,CUST_00003,5.0,5.0,5.0,15.0,15.0,Champions
3,CUST_00004,5.0,5.0,5.0,15.0,15.0,Champions
4,CUST_00005,5.0,5.0,5.0,15.0,15.0,Champions


## 4. Kiểm chứng tính toán trên 1 khách hàng mẫu (`CUST_00001`)

In [5]:
sample_cust = "CUST_00001"
raw_cust = df_valid[df_valid["customer_id"] == sample_cust]
pos_cust = raw_cust[raw_cust["total_amount"] >= 0]
neg_cust = raw_cust[raw_cust["total_amount"] < 0]

n_orders = pos_cust['invoice_id'].nunique()
pos_sum = pos_cust['total_amount'].sum()
neg_sum = neg_cust['total_amount'].sum()
print(f"Khách hàng {sample_cust}: {len(pos_cust)} dòng mua ({n_orders} hóa đơn), {len(neg_cust)} dòng hoàn tiền")
print("Ngày mua đầu:", pos_cust["invoice_date"].min(), "| Ngày mua cuối:", pos_cust["invoice_date"].max())
print("Tổng tiền mua (dương):", f"{pos_sum:,.0f}")
print("Tổng tiền hoàn (âm):", f"{neg_sum:,.0f}")

features_df[features_df["customer_id"] == sample_cust].T.rename(columns=lambda _: sample_cust).head(30)

Khách hàng CUST_00001: 3629 dòng mua (3513 hóa đơn), 41 dòng hoàn tiền
Ngày mua đầu: 2025-11-27 00:37:04 | Ngày mua cuối: 2026-11-26 20:54:36
Tổng tiền mua (dương): 4,141,095,000
Tổng tiền hoàn (âm): -35,126,000


,CUST_00001
customer_id,CUST_00001
monetary_sum,4141095000.0
monetary_mean,1141111.87655
monetary_std,1544243.133797
monetary_min,10000.0
monetary_max,20410000.0
monetary_median,654000.0
frequency,3513.0
recency_days,1.12875
tenure_days,365.974259


## 5. Thống kê mô tả toàn bộ các biến số

In [6]:
summary = features_df.drop(columns=["customer_id", "rfm_segment"]).describe().T[["count", "mean", "std", "min", "50%", "max"]]
summary

,count,mean,std,min,50%,max
monetary_sum,9685.0,1.106234e+07,6.053910e+07,16000.000000,4.493000e+06,4.141095e+09
monetary_mean,9685.0,1.113826e+06,8.416785e+05,16000.000000,9.450000e+05,1.953400e+07
monetary_std,9685.0,1.004130e+06,1.083538e+06,0.000000,7.620839e+05,2.710199e+07
monetary_min,9685.0,3.239942e+05,5.569699e+05,4000.000000,1.600000e+05,1.191400e+07
monetary_max,9685.0,2.988582e+06,2.975572e+06,16000.000000,2.150000e+06,4.538800e+07
monetary_median,9685.0,8.406496e+05,7.574715e+05,16000.000000,6.550000e+05,1.953400e+07
frequency,9685.0,9.984409e+00,5.292586e+01,1.000000,4.000000e+00,3.513000e+03
recency_days,9685.0,7.702505e+01,7.833018e+01,1.004086,4.944150e+01,3.658391e+02
tenure_days,9685.0,2.883788e+02,7.917448e+01,1.142419,3.156430e+02,3.659949e+02
active_span_days,9685.0,2.113538e+02,1.122446e+02,0.000000,2.383261e+02,3.648455e+02


## 6. Minh họa chạy trực tiếp từng hàm lẻ bên trong package

Thay vì gọi qua class tổng hợp `RFMFeatureEngineer`, phần này gọi trực tiếp từng hàm con trong `config.py`, `validators.py`, `features.py` và `utils.py` để thấy rõ luồng xử lý từng bước.

### 6.1. Cấu hình (`RFMConfig`) & Tiền xử lý (`validate_and_prepare`)

In [7]:
from rfm_fe.config import RFMConfig
from rfm_fe.validators import validate_and_prepare

cfg = RFMConfig(
    customer_col="customer_id",
    date_col="invoice_date",
    amount_col="total_amount",
    transaction_col="invoice_id",
    category_col="category",
    reference_date="2026-11-28",
    windows=(30, 90, 180, 365),
    handle_negative_amount="separate",
)

clean_df, ref_date, refund_df = validate_and_prepare(df_valid, cfg)
print("ref_date:", ref_date)
print("clean_df (giao dịch >= 0):", clean_df.shape)
print("refund_df (giao dịch < 0):", refund_df.shape)

ref_date: 2026-11-28 00:00:00
clean_df (giao dịch >= 0): (96991, 5)
refund_df (giao dịch < 0): (1016, 5)


### 6.2. Gọi từng hàm tính đặc trưng trong `rfm_fe/features.py`

In [8]:
from rfm_fe.features import (
    compute_base_rfm,
    compute_ratios,
    compute_window_features,
    compute_interpurchase_features,
    compute_temporal_features,
    compute_category_features,
    compute_refund_features,
)

base_df = compute_base_rfm(clean_df, cfg, ref_date)
ratios_df = compute_ratios(base_df)
windows_df = compute_window_features(clean_df, cfg, ref_date, base_df.index)
ipt_df = compute_interpurchase_features(clean_df, cfg, base_df.index)
temporal_df = compute_temporal_features(clean_df, cfg, base_df.index)
category_df = compute_category_features(clean_df, cfg, base_df.index)
refund_feat_df = compute_refund_features(refund_df, base_df, cfg, base_df.index)

step_parts = {
    "compute_base_rfm": base_df,
    "compute_ratios": ratios_df,
    "compute_window_features": windows_df,
    "compute_interpurchase_features": ipt_df,
    "compute_temporal_features": temporal_df,
    "compute_category_features": category_df,
    "compute_refund_features": refund_feat_df,
}

for fn_name, part_df in step_parts.items():
    print(f"=== {fn_name} -> shape={part_df.shape} ===")
    display(part_df.head(3))

=== compute_base_rfm -> shape=(9685, 10) ===


,monetary_sum,monetary_mean,monetary_std,monetary_min,monetary_max,monetary_median,frequency,recency_days,tenure_days,active_span_days
customer_id,,,,,,,,,,
CUST_00001,4.141095e+09,1.141112e+06,1.544243e+06,10000.0,20410000.0,654000.0,3513.0,1.128750,365.974259,364.845509
CUST_00002,2.276807e+09,1.122133e+06,1.717743e+06,9000.0,33628000.0,638000.0,1988.0,1.291933,365.923889,364.631956
CUST_00003,1.495749e+09,1.046710e+06,1.300909e+06,4000.0,15106000.0,670000.0,1401.0,1.035012,365.830093,364.795081


=== compute_ratios -> shape=(9685, 7) ===


,avg_order_value,monetary_cv,monetary_range,frequency_per_month,monetary_per_month,recency_to_tenure_ratio,is_repeat_customer
customer_id,,,,,,,
CUST_00001,1.178792e+06,1.353279,20400000.0,287.971073,3.394579e+08,0.003084,1.0
CUST_00002,1.145275e+06,1.530784,33619000.0,162.984713,1.866623e+08,0.003531,1.0
CUST_00003,1.067630e+06,1.242855,15102000.0,114.889400,1.226593e+08,0.002829,1.0


=== compute_window_features -> shape=(9685, 14) ===


,freq_30d,monetary_sum_30d,monetary_mean_30d,freq_90d,monetary_sum_90d,monetary_mean_90d,freq_180d,monetary_sum_180d,monetary_mean_180d,freq_365d,monetary_sum_365d,monetary_mean_365d,freq_velocity_30d_vs_365d,monetary_velocity_30d_vs_365d
customer_id,,,,,,,,,,,,,,
CUST_00001,265.0,293911000.0,1.109098e+06,874.0,986835000.0,1.115068e+06,1719.0,1.978084e+09,1.126472e+06,3502.0,4.126757e+09,1.140618e+06,0.920664,0.866520
CUST_00002,164.0,168221000.0,1.025738e+06,508.0,525547000.0,1.030484e+06,975.0,1.057270e+09,1.073371e+06,1979.0,2.269705e+09,1.123616e+06,1.008253,0.901742
CUST_00003,125.0,164873000.0,1.318984e+06,354.0,403559000.0,1.136786e+06,715.0,7.994560e+08,1.105748e+06,1394.0,1.491050e+09,1.048558e+06,1.090985,1.345330


... (4 outputs left out)

=== compute_category_features -> shape=(9685, 3) ===


,category_nunique,category_top_share,category_entropy
customer_id,,,
CUST_00001,5.0,0.209149,2.321316
CUST_00002,5.0,0.203549,2.321829
CUST_00003,5.0,0.214136,2.319902


=== compute_refund_features -> shape=(9685, 3) ===


,refund_count,refund_amount_sum,refund_rate
customer_id,,,
CUST_00001,41.0,35126000.0,0.011536
CUST_00002,22.0,27873000.0,0.010945
CUST_00003,13.0,8459000.0,0.009194


### 6.3. Gọi từng hàm tiện ích trong `rfm_fe/utils.py`

In [9]:
from rfm_fe.utils import (
    safe_divide,
    signed_log1p,
    fit_quantile_edges,
    apply_quantile_score,
    assign_rfm_segment,
)

# 1. Học ngưỡng phân vị & chấm điểm R, F, M
r_edges = fit_quantile_edges(base_df["recency_days"], cfg.n_quantiles)
f_edges = fit_quantile_edges(base_df["frequency"], cfg.n_quantiles)
m_edges = fit_quantile_edges(base_df["monetary_sum"], cfg.n_quantiles)
print("Ngưỡng phân vị Recency:", r_edges.round(2))
print("Ngưỡng phân vị Frequency:", f_edges.round(2))
print("Ngưỡng phân vị Monetary:", m_edges.round(0))

r_score = apply_quantile_score(base_df["recency_days"], r_edges, cfg.n_quantiles, reverse=True)
f_score = apply_quantile_score(base_df["frequency"], f_edges, cfg.n_quantiles, reverse=False)
m_score = apply_quantile_score(base_df["monetary_sum"], m_edges, cfg.n_quantiles, reverse=False)
segments = assign_rfm_segment(r_score, f_score, m_score, cfg.n_quantiles)

scores_preview = pd.DataFrame({
    "recency_days": base_df["recency_days"].round(1),
    "frequency": base_df["frequency"],
    "monetary_sum": base_df["monetary_sum"],
    "r_score": r_score,
    "f_score": f_score,
    "m_score": m_score,
    "rfm_segment": segments,
})
display(scores_preview.head(5))

# 2. Kiểm tra safe_divide & signed_log1p
num = pd.Series([100.0, 50.0, -25.0])
den = pd.Series([4.0, 0.0, 5.0])
print("safe_divide([100, 50, -25], [4, 0, 5]):", safe_divide(num, den).tolist())
print("signed_log1p([100, 0, -25]):", signed_log1p(pd.Series([100.0, 0.0, -25.0])).round(4).tolist())

Ngưỡng phân vị Recency: [ 13.75  34.43  68.16 131.2 ]
Ngưỡng phân vị Frequency: [2. 4. 5. 9.]
Ngưỡng phân vị Monetary: [ 1637800.  3376200.  5840800. 11305200.]


,recency_days,frequency,monetary_sum,r_score,f_score,m_score,rfm_segment
customer_id,,,,,,,
CUST_00001,1.1,3513.0,4.141095e+09,5.0,5.0,5.0,Champions
CUST_00002,1.3,1988.0,2.276807e+09,5.0,5.0,5.0,Champions
CUST_00003,1.0,1401.0,1.495749e+09,5.0,5.0,5.0,Champions
CUST_00004,1.2,1175.0,1.280396e+09,5.0,5.0,5.0,Champions
CUST_00005,1.4,925.0,9.846000e+08,5.0,5.0,5.0,Champions


safe_divide([100, 50, -25], [4, 0, 5]): [25.0, 0.0, -5.0]
signed_log1p([100, 0, -25]): [4.6151, 0.0, -3.2581]
